In [7]:
from pathlib import Path

ROOT = Path.cwd().parents[1]
ROOT

PosixPath('/Users/yangjaehoon/Desktop/StockLens')

In [3]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.feature_selection import RFE

In [8]:
data_path = ROOT / "data/processed/ml_dataset.csv"

df = pd.read_csv(data_path)

df["trade_date"] = pd.to_datetime(df["trade_date"])

df.head()

,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


In [9]:
df.columns.tolist()

['stock_code',
 'trade_date',
 'return_1d',
 'return_5d',
 'return_10d',
 'return_20d',
 'intraday_return',
 'high_low_range',
 'gap',
 'sma_5',
 'sma_20',
 'sma_60',
 'price_to_sma_5',
 'price_to_sma_20',
 'price_to_sma_60',
 'rsi_14',
 'roc_10',
 'roc_20',
 'macd',
 'macd_signal',
 'macd_hist',
 'volatility_5',
 'volatility_20',
 'atr_14',
 'volume_change_1d',
 'volume_sma_20',
 'volume_ratio_20',
 'target_return_5d']

In [10]:
feature_cols = [
    "return_1d",
    "return_5d",
    "return_10d",
    "return_20d",
    "intraday_return",
    "high_low_range",
    "gap",
    "sma_5",
    "sma_20",
    "sma_60",
    "price_to_sma_5",
    "price_to_sma_20",
    "price_to_sma_60",
    "rsi_14",
    "roc_10",
    "roc_20",
    "macd",
    "macd_signal",
    "macd_hist",
    "volatility_5",
    "volatility_20",
    "atr_14",
    "volume_change_1d",
    "volume_sma_20",
    "volume_ratio_20"
]

X = df[feature_cols]
y = df["target_return_5d"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2685, 25)
y shape: (2685,)


In [12]:
rfe_estimator = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

n_features_to_select = [5, 10, 15, 20, 25]
step = [1, 0.1]

In [13]:
rfe_estimator = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=5,
    step=1
)

rfe.fit(X, y)

,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).","XGBRegressor(...ree=None, ...)"
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",5
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
estimator_ estimator_: ``Estimator`` instanceThe fitted estimator used to select features.,XGBRegressor,"XGBRegressor(...ree=None, ...)"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](25,)","['return_1d','return_5d','return_10d',...,'volume_change_1d', 'volume_sma_20','volume_ratio_20']"
n_features_ n_features_: intThe number of selected features.,int64,np.int64(5)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,25
"ranking_ ranking_: ndarray of shape (n_features,)The feature ranking, such that ``ranking_[i]`` corresponds to theranking position of the i-th feature. Selected (i.e., estimatedbest) features are assigned rank 1.","ndarray[int64](25,)","[21, 6,16,...,19, 8,18]"


In [14]:
selected_features = X.columns[rfe.support_].tolist()

selected_features

['sma_20', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']

In [15]:
rfe_results = []

rfe_feature_counts = [5, 10, 15, 20, 25]
rfe_steps = [1, 0.1]

for n_features in rfe_feature_counts:
    for step in rfe_steps:

        rfe_estimator = XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror"
        )

        rfe = RFE(
            estimator=rfe_estimator,
            n_features_to_select=n_features,
            step=step
        )

        rfe.fit(X, y)

        selected_features = X.columns[rfe.support_].tolist()

        rfe_results.append({
            "n_features_to_select": n_features,
            "step": step,
            "selected_features": selected_features
        })

        print(
            f"n_features={n_features}, "
            f"step={step} → {selected_features}"
        )

n_features=5, step=1 → ['sma_20', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']
n_features=5, step=0.1 → ['sma_20', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']
n_features=10, step=1 → ['return_5d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'roc_20', 'macd', 'macd_hist', 'volatility_20', 'atr_14']
n_features=10, step=0.1 → ['return_5d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'rsi_14', 'macd', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=15, step=1 → ['return_5d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'rsi_14', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=15, step=0.1 → ['return_5d', 'high_low_range', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=20, step=1 → ['return_5d', 'return_10d', 'return_20d', 'high_low_ra

In [16]:
for result in rfe_results:
    print(
        f"n_features={result['n_features_to_select']}, "
        f"step={result['step']}:"
    )
    print(result["selected_features"])
    print()

n_features=5, step=1:
['sma_20', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']

n_features=5, step=0.1:
['sma_20', 'sma_60', 'price_to_sma_60', 'volatility_20', 'atr_14']

n_features=10, step=1:
['return_5d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'roc_20', 'macd', 'macd_hist', 'volatility_20', 'atr_14']

n_features=10, step=0.1:
['return_5d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'rsi_14', 'macd', 'volatility_20', 'atr_14', 'volume_sma_20']

n_features=15, step=1:
['return_5d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_60', 'rsi_14', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_sma_20']

n_features=15, step=0.1:
['return_5d', 'high_low_range', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_sma_20']

n_features=20, step=1:
['return_5d', 'return_10d', 'return_20d', 'high_low_ran

In [17]:
rfe_results_df = pd.DataFrame(rfe_results)

rfe_results_df

,n_features_to_select,step,selected_features
0,5,1.0,"[sma_20, sma_60, price_to_sma_60, volatility_2..."
1,5,0.1,"[sma_20, sma_60, price_to_sma_60, volatility_2..."
2,10,1.0,"[return_5d, sma_5, sma_20, sma_60, price_to_sm..."
3,10,0.1,"[return_5d, sma_5, sma_20, sma_60, price_to_sm..."
4,15,1.0,"[return_5d, return_20d, sma_5, sma_20, sma_60,..."
5,15,0.1,"[return_5d, high_low_range, sma_5, sma_20, sma..."
6,20,1.0,"[return_5d, return_10d, return_20d, high_low_r..."
7,20,0.1,"[return_5d, return_10d, return_20d, high_low_r..."
8,25,1.0,"[return_1d, return_5d, return_10d, return_20d,..."
9,25,0.1,"[return_1d, return_5d, return_10d, return_20d,..."


In [18]:
rfe_results_df.to_csv(
    ROOT / "data/processed/rfe_xgboost_feature_sets.csv",
    index=False
)

In [19]:
xgb_params = {
    "n_estimators": [100, 300, 500],
    "max_depth": [2, 3, 5],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

In [20]:
print(df[["return_5d", "target_return_5d"]].head(10))

   return_5d  target_return_5d
0   0.094233          0.103529
1   0.112261          0.086047
2   0.146102          0.069820
3   0.065060          0.058824
4   0.072115          0.000000
5   0.103529         -0.040512
6   0.086047          0.014989
7   0.069820         -0.004211
8   0.058824          0.010684
9   0.000000          0.056054


In [21]:
print("return_5d null:", df["return_5d"].isna().sum())
print("target_return_5d null:", df["target_return_5d"].isna().sum())

return_5d null: 0
target_return_5d null: 0


In [22]:
target = "target_return_5d"

train_df = df[
    (df["trade_date"] >= "2024-03-13") &
    (df["trade_date"] <= "2025-12-31")
].copy()

valid_df = df[
    (df["trade_date"] >= "2026-01-01") &
    (df["trade_date"] <= "2026-06-30")
].copy()

features = [
    col
    for col in df.columns
    if col not in ["stock_code", "trade_date", target]
]

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Feature count:", len(features))
print(features)

Train: (1895, 28)
Validation: (600, 28)
Feature count: 25
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


In [23]:
X_train = train_df[features]
y_train = train_df[target]

X_valid = valid_df[features]
y_valid = valid_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (1895, 25)
y_train: (1895,)
X_valid: (600, 25)
y_valid: (600,)


In [24]:
rfe_results = []

rfe_feature_counts = [5, 10, 15, 20, 25]
rfe_steps = [1, 0.1]

for n_features in rfe_feature_counts:
    for step in rfe_steps:

        rfe_estimator = XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror"
        )

        rfe = RFE(
            estimator=rfe_estimator,
            n_features_to_select=n_features,
            step=step
        )

        rfe.fit(X_train, y_train)

        selected_features = X_train.columns[rfe.support_].tolist()

        rfe_results.append({
            "n_features_to_select": n_features,
            "step": step,
            "selected_features": selected_features
        })

        print(
            f"n_features={n_features}, "
            f"step={step} → {selected_features}"
        )

n_features=5, step=1 → ['sma_5', 'sma_60', 'macd_signal', 'volatility_20', 'atr_14']
n_features=5, step=0.1 → ['sma_5', 'sma_60', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=10, step=1 → ['return_20d', 'sma_5', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=10, step=0.1 → ['return_20d', 'sma_5', 'sma_60', 'price_to_sma_60', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=15, step=1 → ['return_5d', 'return_10d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=15, step=0.1 → ['return_5d', 'return_20d', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_sma_20']
n_features=20, step=1 → ['return_5d', 'return_10d', 'return_

In [25]:
from itertools import product

xgb_param_grid = list(product(
    [100, 300, 500],      # n_estimators
    [2, 3, 5],            # max_depth
    [0.03, 0.1],          # learning_rate
    [0.8, 1.0],            # subsample
    [0.8, 1.0]             # colsample_bytree
))

print("XGBoost combinations:", len(xgb_param_grid))
print("Total experiments:", len(rfe_results) * len(xgb_param_grid))

XGBoost combinations: 72
Total experiments: 720


In [26]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []

total = len(rfe_results) * len(xgb_param_grid)
experiment = 0

for rfe_result in rfe_results:

    n_features = rfe_result["n_features_to_select"]
    step = rfe_result["step"]
    selected_features = rfe_result["selected_features"]

    X_tr = X_train[selected_features]
    X_va = X_valid[selected_features]

    for params in xgb_param_grid:

        n_estimators, max_depth, learning_rate, subsample, colsample_bytree = params

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=42,
            objective="reg:squarederror"
        )

        model.fit(X_tr, y_train)

        y_pred = model.predict(X_va)

        rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
        mae = mean_absolute_error(y_valid, y_pred)
        r2 = r2_score(y_valid, y_pred)

        results.append({
            "n_features_to_select": n_features,
            "rfe_step": step,
            "selected_features": ",".join(selected_features),
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        })

        experiment += 1

        if experiment % 10 == 0:
            print(f"Progress: {experiment}/{total}")

Progress: 10/720
Progress: 20/720
Progress: 30/720
Progress: 40/720
Progress: 50/720
Progress: 60/720
Progress: 70/720
Progress: 80/720
Progress: 90/720
Progress: 100/720
Progress: 110/720
Progress: 120/720
Progress: 130/720
Progress: 140/720
Progress: 150/720
Progress: 160/720
Progress: 170/720
Progress: 180/720
Progress: 190/720
Progress: 200/720
Progress: 210/720
Progress: 220/720
Progress: 230/720
Progress: 240/720
Progress: 250/720
Progress: 260/720
Progress: 270/720
Progress: 280/720
Progress: 290/720
Progress: 300/720
Progress: 310/720
Progress: 320/720
Progress: 330/720
Progress: 340/720
Progress: 350/720
Progress: 360/720
Progress: 370/720
Progress: 380/720
Progress: 390/720
Progress: 400/720
Progress: 410/720
Progress: 420/720
Progress: 430/720
Progress: 440/720
Progress: 450/720
Progress: 460/720
Progress: 470/720
Progress: 480/720
Progress: 490/720
Progress: 500/720
Progress: 510/720
Progress: 520/720
Progress: 530/720
Progress: 540/720
Progress: 550/720
Progress: 560/720
P

In [27]:
results_df = pd.DataFrame(results)

print(results_df.shape)
results_df.head()

(720, 11)


,n_features_to_select,rfe_step,selected_features,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,rmse,mae,r2
0,5,1.0,"sma_5,sma_60,macd_signal,volatility_20,atr_14",100,2,0.03,0.8,0.8,0.102682,0.076605,0.037854
1,5,1.0,"sma_5,sma_60,macd_signal,volatility_20,atr_14",100,2,0.03,0.8,1.0,0.102591,0.076478,0.039567
2,5,1.0,"sma_5,sma_60,macd_signal,volatility_20,atr_14",100,2,0.03,1.0,0.8,0.102741,0.076590,0.036746
3,5,1.0,"sma_5,sma_60,macd_signal,volatility_20,atr_14",100,2,0.03,1.0,1.0,0.102738,0.076464,0.036800
4,5,1.0,"sma_5,sma_60,macd_signal,volatility_20,atr_14",100,2,0.10,0.8,0.8,0.104840,0.078289,-0.003013


In [28]:
results_df.sort_values("rmse").head(10)

,n_features_to_select,rfe_step,selected_features,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,rmse,mae,r2
75,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,1.0,0.100622,0.074809,0.076078
73,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,1.0,0.100748,0.074836,0.073761
74,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,0.8,0.100998,0.075342,0.069157
87,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,3,0.10,1.0,1.0,0.101580,0.076647,0.058399
72,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,0.8,0.101589,0.075946,0.058230
97,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",300,2,0.03,0.8,1.0,0.101991,0.076346,0.050767
288,15,1.0,"return_5d,return_10d,return_20d,sma_5,sma_20,s...",100,2,0.03,0.8,0.8,0.102047,0.075877,0.049712
360,15,0.1,"return_5d,return_20d,sma_5,sma_20,sma_60,price...",100,2,0.03,0.8,0.8,0.102103,0.075934,0.048675
504,20,0.1,"return_5d,return_10d,return_20d,intraday_retur...",100,2,0.03,0.8,0.8,0.102185,0.076141,0.047149
76,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.10,0.8,0.8,0.102213,0.076836,0.046633


In [29]:
results_df.sort_values("mae").head(10)

,n_features_to_select,rfe_step,selected_features,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,rmse,mae,r2
75,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,1.0,0.100622,0.074809,0.076078
73,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,1.0,0.100748,0.074836,0.073761
74,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,0.8,0.100998,0.075342,0.069157
288,15,1.0,"return_5d,return_10d,return_20d,sma_5,sma_20,s...",100,2,0.03,0.8,0.8,0.102047,0.075877,0.049712
360,15,0.1,"return_5d,return_20d,sma_5,sma_20,sma_60,price...",100,2,0.03,0.8,0.8,0.102103,0.075934,0.048675
72,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,0.8,0.101589,0.075946,0.058230
579,25,1.0,"return_1d,return_5d,return_10d,return_20d,intr...",100,2,0.03,1.0,1.0,0.102545,0.076062,0.040414
651,25,0.1,"return_1d,return_5d,return_10d,return_20d,intr...",100,2,0.03,1.0,1.0,0.102545,0.076062,0.040414
218,10,0.1,"return_20d,sma_5,sma_60,price_to_sma_60,macd,m...",100,2,0.03,1.0,0.8,0.102276,0.076087,0.045444
146,10,1.0,"return_20d,sma_5,sma_60,price_to_sma_60,macd,m...",100,2,0.03,1.0,0.8,0.102276,0.076087,0.045444


In [30]:
results_df.sort_values("r2", ascending=False).head(10)

,n_features_to_select,rfe_step,selected_features,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,rmse,mae,r2
75,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,1.0,0.100622,0.074809,0.076078
73,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,1.0,0.100748,0.074836,0.073761
74,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,1.0,0.8,0.100998,0.075342,0.069157
87,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,3,0.10,1.0,1.0,0.101580,0.076647,0.058399
72,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.03,0.8,0.8,0.101589,0.075946,0.058230
97,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",300,2,0.03,0.8,1.0,0.101991,0.076346,0.050767
288,15,1.0,"return_5d,return_10d,return_20d,sma_5,sma_20,s...",100,2,0.03,0.8,0.8,0.102047,0.075877,0.049712
360,15,0.1,"return_5d,return_20d,sma_5,sma_20,sma_60,price...",100,2,0.03,0.8,0.8,0.102103,0.075934,0.048675
504,20,0.1,"return_5d,return_10d,return_20d,intraday_retur...",100,2,0.03,0.8,0.8,0.102185,0.076141,0.047149
76,5,0.1,"sma_5,sma_60,volatility_20,atr_14,volume_sma_20",100,2,0.10,0.8,0.8,0.102213,0.076836,0.046633


In [31]:
results_df.head()
results_df.shape

(720, 11)

In [34]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "../../data/processed/filter_results/rfe_xgboost_results.csv",
    index=False
)

print("Saved:", results_df.shape)

Saved: (720, 11)
